<a href="https://colab.research.google.com/github/galitneu/Diabetes/blob/main/01_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation and Cleaning Notebook

## Overview
This notebook performs comprehensive data preparation and cleaning for the **Diabetic Readmission Dataset**. The goal is to prepare the data for machine learning models by handling missing values, creating meaningful features, and optimizing data types.

## Dataset Information
- **Source:** Diabetic patient hospital readmission data
- **Original records:** ~100,000+ hospital encounters
- **Features:** Patient demographics, medical history, medications, diagnoses, and readmission status

## Key Processing Steps
1. **Data Loading** - Import data from Google Drive
2. **Initial Exploration** - Understand data structure and quality
3. **Data Cleaning** - Remove invalid records and handle missing values
4. **Feature Engineering** - Create new meaningful features from existing data
5. **Data Type Optimization** - Convert to appropriate data types for efficiency
6. **Final Export** - Save prepared dataset for modeling

## Target Variable
**`readmitted`** - Indicates whether a patient was readmitted to the hospital:
- `NO` - Patient was not readmitted
- `<30` - Readmitted within 30 days
- `>30` - Readmitted after 30 days

---

## 1. Import Required Libraries

Import all necessary libraries for data manipulation, visualization, and Google Colab integration.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import IPython
import os
#from ydata_profiling import ProfileReport
from google.colab import drive
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 2. Load Data from Google Drive

Mount Google Drive and load the diabetic dataset along with mapping files.

In [2]:



# Mount the Google Drive to the '/content/drive' directory
drive.mount('/content/drive')
# Print a success message indicating the Drive is mounted
print("✅ Google Drive mounted successfully!")


# The default is the main 'My Drive' folder.
# Define the base path variable pointing to 'My Drive'
base_drive_path = '/content/drive/My Drive/'

# Main file
# Construct the full path to the main data file
file1_path = os.path.join(base_drive_path, 'diabetic_data.csv')

# Key for categories
# Construct the full path to the mapping file
file2_path = os.path.join(base_drive_path, 'IDS_mapping.csv')

# Open the mapping file in read ('r') mode
with open(file2_path, 'r') as f:
    # Read the entire content of the file into a variable
    mapping_content = f.read()

# Start a try-except block to handle potential file reading errors
try:
    df1 = pd.read_csv(file1_path,index_col=0)
    print("The file was successfully uploaded")
# Catch the FileNotFoundError if the file doesn't exist
except FileNotFoundError as e:
    print(f"Error: The file was not found - {e}. Please make sure the files are in the correct directory.")
    exit()

df1

Mounted at /content/drive
✅ Google Drive mounted successfully!
The file was successfully uploaded


,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
encounter_id,,,,,,,,,,,,,,,,,,,,,
2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,...,No,No,No,No,No,No,No,No,No,NO
149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,...,No,Up,No,No,No,No,No,Ch,Yes,>30
64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,...,No,No,No,No,No,No,No,No,Yes,NO
500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,...,No,Up,No,No,No,No,No,Ch,Yes,NO
16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,MC,...,No,Down,No,No,No,No,No,Ch,Yes,>30
443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,MC,...,No,Steady,No,No,No,No,No,No,Yes,NO
443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,MC,...,No,Down,No,No,No,No,No,Ch,Yes,NO


## 3. Initial Data Exploration

Reset index and examine the basic structure of the dataset.

In [3]:
df1.reset_index(inplace=True)


In [4]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## 4. Data Cleaning - Remove Invalid Records

Remove records for patients who expired or went to hospice (won't be readmitted).

In [5]:
# Rows for people who went to hospice or expired are not relevant
# because they won't be readmitted. So, we delete such columns.

# Define the list of 'discharge_disposition_id' values that correspond to 'Expired' or 'Hospice'
expired_ids = [11, 19, 20, 21]
# Get the total number of records before filtering
before_count = len(df1)

# Filter the DataFrame, keeping only the rows WHERE the 'discharge_disposition_id' is NOT IN (~) the 'expired_ids' list
df1 = df1[~df1['discharge_disposition_id'].isin(expired_ids)]

# Get the new count of records after filtering
after_count = len(df1)
# Calculate the total number of removed records
removed = before_count - after_count

# Print a summary of the removal process
print(f"\nRemoving Expired/Hospice Records:")
print(f"  Before: {before_count:,} records")
print(f"  After: {after_count:,} records")
print(f"  Removed: {removed:,} records ({removed/before_count*100:.2f}%)")


Removing Expired/Hospice Records:
  Before: 101,766 records
  After: 100,114 records
  Removed: 1,652 records (1.62%)


## 5. Data Type Conversions

Convert object columns to string type for better memory efficiency.

In [6]:
# Convert columns defined as 'object' to 'string'
# Select all columns that currently have the 'object' data type
string_columns = df1.select_dtypes(include=['object']).columns

# Loop through each column identified as an 'object' type
for col in string_columns:
    # Remove any leading or trailing whitespace from the values in the column
    df1[col]=df1[col].str.strip()
    # Convert the column's data type from 'object' to the pandas 'string' type
    df1[col]=df1[col].astype('string')

df1.info()


<class 'pandas.core.frame.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              100114 non-null  int64 
 1   patient_nbr               100114 non-null  int64 
 2   race                      100114 non-null  string
 3   gender                    100114 non-null  string
 4   age                       100114 non-null  string
 5   weight                    100114 non-null  string
 6   admission_type_id         100114 non-null  int64 
 7   discharge_disposition_id  100114 non-null  int64 
 8   admission_source_id       100114 non-null  int64 
 9   time_in_hospital          100114 non-null  int64 
 10  payer_code                100114 non-null  string
 11  medical_specialty         100114 non-null  string
 12  num_lab_procedures        100114 non-null  int64 
 13  num_procedures            100114 non-null  int64 
 14  num_medic

## 6. Missing Values Analysis

Analyze missing values and '?' symbols across all columns.

In [7]:
df_cleaned = df1

# Check the percentage of values that are null, '?', or 0
# Iterate over every column in the DataFrame
for col in df1.columns:
        # Count the sum of nulls (NaN), '?' strings, and zero (0) values
        missing = df1[col].isnull().sum() + (df1[col] == '?').sum() + (df1[col] == 0).sum()

        # Calculate what percentage of the total rows this 'missing' count represents
        pct = missing / len(df1) * 100

        # If it's above 0, print the name
        # If the percentage of 'missing' values is greater than 0
        if pct > 0:
            # Print the column name and its 'missing' percentage
            print(f"{col}: {pct:.2f}%")

            # If it's over 90%, delete the column
            if pct > 90:
                # Drop this column from the 'df_cleaned' DataFrame
                df_cleaned = df_cleaned.drop(columns=col)
                print(f"{col} dropped")

race: 2.24%
weight: 96.85%
weight dropped
payer_code: 39.55%
medical_specialty: 49.07%
num_procedures: 46.04%
number_outpatient: 83.53%
number_emergency: 88.82%
number_inpatient: 66.60%
diag_1: 0.02%
diag_2: 0.36%
diag_3: 1.42%
max_glu_serum: 94.78%
max_glu_serum dropped
A1Cresult: 83.14%


## 7. Handle Target Variable (readmitted)

Process and analyze the target variable distribution.

In [8]:
# ========================================================================
# Handling 'readmitted' - the dependent variable
# ========================================================================

print("=" * 70)
print("Handling 'readmitted' - the dependent variable")
print("=" * 70)

# Check original distribution
print("\nOriginal readmitted distribution:")
readmit_counts = df_cleaned['readmitted'].value_counts()
for category, count in readmit_counts.items():
    pct = (count / len(df_cleaned)) * 100
    print(f"  {category:5s}: {count:7,} ({pct:5.2f}%)")

# Create a binary variable - readmitted within 30 days
# The goal is to predict if the person will be readmitted within 30 days
# So, reduce the categories to 0 (No) and 1 (Yes)
print("\n" + "=" * 70)
print("Binary encoding:")
print("  <30  → 1 (readmitted within 30 days)")
print("  >30  → 0 (not readmitted within 30 days)")
print("  NO   → 0 (not readmitted)")
print("=" * 70)

# Create the new binary column. It will be True (1) only if the value is '<30', and False (0) otherwise.
df_cleaned['readmitted_binary'] = (df_cleaned['readmitted'] == '<30').astype(bool)

# Final distribution
readmitted = df_cleaned['readmitted_binary'].sum()
not_readmitted = (df_cleaned['readmitted_binary'] == 0).sum()

print(f"\nreadmitted_binary distribution:")
print(f"  Readmitted (1): {readmitted:7,} ({readmitted/len(df_cleaned)*100:.2f}%)")
print(f"  Not Readmitted (0): {not_readmitted:7,} ({not_readmitted/len(df_cleaned)*100:.2f}%)")

# Check balance
ratio = readmitted / not_readmitted
print(f"\nImbalance ratio: 1:{ratio:.2f}")

print("\n" + "=" * 70)
print("✓ Column created: readmitted_binary")
print("=" * 70)

Handling 'readmitted' - the dependent variable

Original readmitted distribution:
  NO   :  53,212 (53.15%)
  >30  :  35,545 (35.50%)
  <30  :  11,357 (11.34%)

Binary encoding:
  <30  → 1 (readmitted within 30 days)
  >30  → 0 (not readmitted within 30 days)
  NO   → 0 (not readmitted)

readmitted_binary distribution:
  Readmitted (1):  11,357 (11.34%)
  Not Readmitted (0):  88,757 (88.66%)

Imbalance ratio: 1:0.13

✓ Column created: readmitted_binary


## 8. Process Categorical Variables

Clean and standardize categorical variables like race and gender.

In [9]:
# ========================================================================
# Handling simple variables
# ========================================================================

# 1. race
# Replace the '?' value with 'Unknown' to make the category explicit
df_cleaned['race'] = df_cleaned['race'].replace('?', 'Unknown')
print(df_cleaned['race'].value_counts())

race
Caucasian          74845
AfricanAmerican    18888
Unknown             2239
Hispanic            2024
Other               1486
Asian                632
Name: count, dtype: Int64


In [10]:

# Print a main header for this analysis section
print("=" * 80)
print("RACE GROUP STATISTICS ANALYSIS")
print("=" * 80)

# Use existing age_numeric column if it exists, otherwise create it
if 'age_numeric' not in df_cleaned.columns:
    # Define a dictionary to map age ranges to numeric midpoints
    age_map = {
        '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
        '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
    }
    # Create the 'age_numeric' column by applying the map
    df_cleaned['age_numeric'] = df_cleaned['age'].map(age_map)



# Create a temporary helper column to flag readmissions (<30 or >30)
# This allows us to use 'sum' within the groupby aggregation
df_cleaned['temp_readmit_flag'] = df_cleaned['readmitted'].isin(['<30', '>30'])

# Perform all aggregations in a single groupby operation
# We use .agg() with NamedAgg for clear output column names
stats_df = df_cleaned.groupby('race').agg(
    Count=pd.NamedAgg(column='race', aggfunc='size'),
    Avg_Age=pd.NamedAgg(column='age_numeric', aggfunc='mean'),
    Avg_Meds=pd.NamedAgg(column='num_medications', aggfunc='mean'),
    Avg_Days=pd.NamedAgg(column='time_in_hospital', aggfunc='mean'),
    Avg_Diag=pd.NamedAgg(column='number_diagnoses', aggfunc='mean'),
    # Sum the temporary flags to count readmissions
    Readmit_Count_Temp=pd.NamedAgg(column='temp_readmit_flag', aggfunc='sum')
)

# Calculate the readmission percentage using the total Count and the temp count
stats_df['Readmit_Pct'] = (stats_df['Readmit_Count_Temp'] / stats_df['Count'] * 100).fillna(0)

# Clean up: drop the temporary helper column from the main DataFrame
df_cleaned = df_cleaned.drop(columns=['temp_readmit_flag'])

print("\nRace Group Statistics:\n")
# Print the table header
print("-" * 80)
print(f"{'Race':<20} {'Count':>8} {'Avg Age':>10} {'Readmit%':>10} "
      f"{'Avg Meds':>10} {'Avg Days':>10} {'Avg Diag':>10}")
print("-" * 80)

# Iterate through the *already calculated* stats_df to print each row
# .groupby() automatically sorts the index ('race')
for race, row in stats_df.iterrows():
    print(f"{race:<20} {row['Count']:>8,} {row['Avg_Age']:>10.1f} {row['Readmit_Pct']:>9.2f}% "
          f"{row['Avg_Meds']:>10.2f} {row['Avg_Days']:>10.2f} {row['Avg_Diag']:>10.2f}")

print("-" * 80)


# Drop the temporary count column used for the percentage calculation
stats_df = stats_df.drop(columns=['Readmit_Count_Temp'])

# Add the 'Percentage' column (percentage of total records)
stats_df['Percentage'] = (stats_df['Count'] / len(df_cleaned) * 100).round(2)

# Sort by 'Count' descending, just like the original script
stats_df = stats_df.sort_values('Count', ascending=False)

# Reset the index to turn 'race' from an index back into a regular column
# The resulting stats_df is now identical to the one from the original script
stats_df = stats_df.reset_index()



RACE GROUP STATISTICS ANALYSIS

Race Group Statistics:

--------------------------------------------------------------------------------
Race                    Count    Avg Age   Readmit%   Avg Meds   Avg Days   Avg Diag
--------------------------------------------------------------------------------
AfricanAmerican      18,888.0       60.5     46.53%      15.31       4.50       7.09
Asian                   632.0       66.1     35.76%      13.21       3.97       7.03
Caucasian            74,845.0       67.4     47.72%      16.25       4.38       7.53
Hispanic              2,024.0       58.8     42.19%      14.00       4.06       6.92
Other                 1,486.0       62.5     39.77%      15.14       4.27       7.17
Unknown               2,239.0       66.4     32.43%      15.77       4.29       6.67
--------------------------------------------------------------------------------


In [11]:
# 2. gender
# Standardize the 'Unknown/Invalid' value to simply 'Unknown'
df_cleaned['gender'] = df_cleaned['gender'].replace('Unknown/Invalid', 'Unknown')

# 3. change → change_binary
# Create a binary variable from the 'change' column (referring to whether there was a change in medications)
# Map 'Ch' (Changed) to 1 and 'No' (No change) to 0
df_cleaned['change_binary'] = (df_cleaned['change'] == 'Ch').astype(int)

## 9. Payer Code Processing

Group and categorize insurance payer codes.

In [12]:
#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Handling the 'payer_code' variable
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

# Check its distribution
# Print the top 20 most frequent values in the 'payer_code' column
print(df_cleaned['payer_code'].value_counts().head(20))

payer_code
?     39591
MC    31739
HM     6218
SP     4956
BC     4625
MD     3492
CP     2498
UN     2425
CM     1907
OG     1023
PO      586
DM      546
CH      144
WC      135
OT       94
MP       79
SI       55
FR        1
Name: count, dtype: Int64


In [13]:

print("=" * 80)
print("Grouping Payer Code")
print("=" * 80)

# =============================================================================
# Direct mapping of payer's codes to categories
# =============================================================================

payer_mapping = {
    # Unknown / Missing
    '?': 'Unknown',
    'UN': 'Unknown',

    # Medicare (Public insurance for seniors)
    'MC': 'Medicare',
    'DM': 'Medicare',  # Dual Medicare/Medicaid
    'MP': 'Medicare',  # Medicare Part

    # Medicaid (Public insurance for eligible individuals)
    'MD': 'Medicaid',

    # Commercial (Private/commercial insurance)
    'HM': 'Commercial',  # HMO
    'CM': 'Commercial',
    'CP': 'Commercial',
    'BC': 'Commercial',  # Blue Cross
    'PO': 'Commercial',  # Private

    # Self Pay
    'SP': 'Self_Pay',

    # Government Other
    'OG': 'Government_Other',
    'CH': 'Government_Other',  # CHAMPUS/TRICARE (Military)
    'WC': 'Government_Other',  # Workers Compensation

    # Free Care
    'FR': 'Other', # Only one entry like this
}

# Group according to the dictionary
# .fillna('Other') will catch any codes not in the map (like 'SI', 'OT', etc.)
df_cleaned['payer_grouped'] = df_cleaned['payer_code'].map(payer_mapping).fillna('Unknown')


print("\nPayer Grouped Distribution:")
print("=" * 80)

# Get the total count for percentage calculation
total = len(df_cleaned)

# Group by the new category
summary = (
    df_cleaned.groupby('payer_grouped')
    .agg(
        # Count the size of each group
        Count=('payer_grouped', 'size'),
        # Get a list of the unique original codes that were mapped to this group
        Original_Codes=('payer_code', lambda x: ', '.join(sorted(x.unique())))
    )
    .sort_values(by='Count', ascending=False) # Sort by count
)
# Calculate the percentage for each group
summary['Percent'] = (summary['Count'] / total) * 100

print(f"{'Group':20s} | {'Count':>10s} | {'Percent':>7s} | Original Codes")
print("-" * 80)
for group, row in summary.iterrows():
    print(f"{group:20s} | {row['Count']:>10,} | {row['Percent']:>6.2f}% | [{row['Original_Codes']}]")

print(f"\nTotal categories: {df_cleaned['payer_grouped'].nunique()}")

Grouping Payer Code

Payer Grouped Distribution:
Group                |      Count | Percent | Original Codes
--------------------------------------------------------------------------------
Unknown              |     42,165 |  42.12% | [?, OT, SI, UN]
Medicare             |     32,364 |  32.33% | [DM, MC, MP]
Commercial           |     15,834 |  15.82% | [BC, CM, CP, HM, PO]
Self_Pay             |      4,956 |   4.95% | [SP]
Medicaid             |      3,492 |   3.49% | [MD]
Government_Other     |      1,302 |   1.30% | [CH, OG, WC]
Other                |          1 |   0.00% | [FR]

Total categories: 7


## 10. A1C Result Processing

Handle A1C test results and their categories.

In [14]:
# ========================================================================
# Handling A1Cresult
#
# A1C is an important diabetes metric. Because of this, despite 83% missing values,
# this column will be kept and not dropped.
#
# ========================================================================


# Check before
print("\nBefore processing:")
print(f"  NaN (empty): {df_cleaned['A1Cresult'].isna().sum():,}")
print(f"  '?' (if any): {(df_cleaned['A1Cresult'] == '?').sum():,}")
total_missing = df_cleaned['A1Cresult'].isna().sum()
print(f"  Total missing: {total_missing:,} ({total_missing/len(df_cleaned)*100:.1f}%)")

# Convert NaN to 'Not_Measured' (text)
df_cleaned['A1Cresult'] = df_cleaned['A1Cresult'].fillna('Not_Measured')

# If there are '?' - also convert
if (df_cleaned['A1Cresult'] == '?').any():
    df_cleaned['A1Cresult'] = df_cleaned['A1Cresult'].replace('?', 'Not_Measured')

# Convert to category dtype
df_cleaned['A1Cresult'] = df_cleaned['A1Cresult'].astype('category')

# Final distribution
print("\nAfter processing - A1Cresult distribution:")
for value, count in df_cleaned['A1Cresult'].value_counts().items():
    pct = count / len(df_cleaned) * 100
    print(f"  {value:15s}: {count:7,} ({pct:5.2f}%)")


Before processing:
  NaN (empty): 83,238
  '?' (if any): 0
  Total missing: 83,238 (83.1%)

After processing - A1Cresult distribution:
  Not_Measured   :  83,238 (83.14%)
  >8             :   8,151 ( 8.14%)
  Norm           :   4,941 ( 4.94%)
  >7             :   3,784 ( 3.78%)


In [15]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 52 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   encounter_id              100114 non-null  int64   
 1   patient_nbr               100114 non-null  int64   
 2   race                      100114 non-null  string  
 3   gender                    100114 non-null  string  
 4   age                       100114 non-null  string  
 5   admission_type_id         100114 non-null  int64   
 6   discharge_disposition_id  100114 non-null  int64   
 7   admission_source_id       100114 non-null  int64   
 8   time_in_hospital          100114 non-null  int64   
 9   payer_code                100114 non-null  string  
 10  medical_specialty         100114 non-null  string  
 11  num_lab_procedures        100114 non-null  int64   
 12  num_procedures            100114 non-null  int64   
 13  num_medications           100114 n

## 11. Diagnosis Code Grouping

Categorize ICD-9 diagnosis codes into disease categories.

In [16]:
"""
Diagnosis Grouping Script for Diabetes Readmission Dataset

This script groups ICD-9 diagnosis codes into clinical categories and creates
additional features to help predict hospital readmission.

Input: DataFrame with diag_1, diag_2, diag_3 columns (ICD-9 codes)
Output:
  - Category columns (10 categories + Other)
  - Severity columns (0-3)
  - Diabetes position column (0/1/2/3)
"""


# =============================================================================
# STEP 1: Analyze Original ICD-9 Categories Distribution (Before Grouping)
# =============================================================================

print("=" * 80)
print("STEP 1: Original ICD-9 Category Distribution Analysis")
print("=" * 80)

def get_original_icd9_category(code):
    """
    Map diagnosis code to original ICD-9 chapters (19 categories)
    This is the official ICD-9 classification system

    ICD-9 Chapters:
    1. Infectious (001-139)
    2. Neoplasms (140-239)
    3. Endocrine/Metabolic/Immunity (240-279)
    4. Blood (280-289)
    5. Mental (290-319)
    6. Nervous System (320-389)
    7. Circulatory (390-459)
    8. Respiratory (460-519)
    9. Digestive (520-579)
    10. Genitourinary (580-629)
    11. Pregnancy (630-679)
    12. Skin (680-709)
    13. Musculoskeletal (710-739)
    14. Congenital (740-759)
    15. Perinatal (760-779)
    16. Symptoms/Signs (780-799)
    17. Injury/Poisoning (800-999)
    18. E codes - External Causes (E800-E999)
    19. V codes - Supplementary (V01-V91)
    """
    if pd.isna(code) or code == '?' or str(code).strip() == '':
        return 'Missing'

    code_str = str(code).strip().upper()

    # Special codes
    if code_str.startswith('V'):
        return 'V_codes'
    if code_str.startswith('E'):
        return 'E_codes'

    # Convert to numeric
    code_str = code_str.replace('.', '')
    try:
        code_num = float(code_str)
    except:
        return 'Invalid'

    # Map to ICD-9 chapters
    if 1 <= code_num < 140:
        return 'Infectious'
    elif 140 <= code_num < 240:
        return 'Neoplasms'
    elif 240 <= code_num < 280:
        return 'Endocrine_Metabolic'
    elif 280 <= code_num < 290:
        return 'Blood'
    elif 290 <= code_num < 320:
        return 'Mental'
    elif 320 <= code_num < 390:
        return 'Nervous'
    elif 390 <= code_num < 460:
        return 'Circulatory'
    elif 460 <= code_num < 520:
        return 'Respiratory'
    elif 520 <= code_num < 580:
        return 'Digestive'
    elif 580 <= code_num < 630:
        return 'Genitourinary'
    elif 630 <= code_num < 680:
        return 'Pregnancy'
    elif 680 <= code_num < 710:
        return 'Skin'
    elif 710 <= code_num < 740:
        return 'Musculoskeletal'
    elif 740 <= code_num < 760:
        return 'Congenital'
    elif 760 <= code_num < 780:
        return 'Perinatal'
    elif 780 <= code_num < 800:
        return 'Symptoms_Signs'
    elif 800 <= code_num < 1000:
        return 'Injury_Poisoning'
    else:
        return 'Other'


def analyze_original_icd9_distribution(df, diag_cols=['diag_1', 'diag_2', 'diag_3']):
    """
    Analyze the distribution of ICD-9 categories before our custom grouping
    Shows how diagnoses are distributed across the 19 official ICD-9 chapters
    """
    for diag_col in diag_cols:
        print(f"\n{'='*80}")
        print(f"{diag_col.upper()} - Original ICD-9 Category Distribution")
        print('='*80)

        # Get ICD-9 categories
        icd9_categories = df[diag_col].apply(get_original_icd9_category)
        cat_counts = icd9_categories.value_counts()
        total = len(df)

        print(f"\nTotal records: {total:,}")
        print(f"Number of ICD-9 categories: {len(cat_counts)}")

        print(f"\n{'ICD-9 Category':<25s} {'Count':>10s} {'Percentage':>12s}")
        print("-" * 50)

        for cat, count in cat_counts.sort_values(ascending=False).items():
            pct = count / total * 100
            print(f"{cat:<25s} {count:>10,} {pct:>11.2f}%")

# Run the analysis
analyze_original_icd9_distribution(df_cleaned)


# =============================================================================
# STEP 2: Define Grouping Functions
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: Define Grouping Functions")
print("=" * 80)

def is_diabetes(code):
    """
    Check if a diagnosis code is diabetes (250.xx)
    """
    if pd.isna(code) or code == '?':
        return False

    code_str = str(code).strip().upper()

    # V and E codes are not diabetes
    if code_str.startswith('V') or code_str.startswith('E'):
        return False

    # Convert to numeric FIRST (keep decimal), then check range
    try:
        code_num = float(code_str)
        return 250 <= code_num < 251
    except:
        return False


def categorize_diagnosis(code):
    """
    Group ICD-9 diagnosis codes into 10 clinical categories + Other

    Categories (based on ICD-9 chapters + clinical importance):
    1. Circulatory (390-459, 785)
    2. Respiratory (460-519, 786)
    3. Genitourinary (580-629, 788)
    4. Endocrine (240-279) - includes Diabetes 250.xx
    5. Digestive (520-579, 787)
    6. Injury (800-999)
    7. Symptoms_Signs (780-799)
    8. Musculoskeletal (710-739)
    9. Neoplasms (140-239)
    10. Infectious (001-139)
    11. Other (all other codes)

    Args:
        code: ICD-9 diagnosis code

    Returns:
        str: Category name
    """
    if pd.isna(code) or code == '?' or str(code).strip() == '':
        return 'Other'

    code_str = str(code).strip().upper()

    # V codes - Supplementary Classification
    if code_str.startswith('V'):
        return 'Other'

    # E codes - External causes
    if code_str.startswith('E'):
        return 'Other'

    # Convert to numeric
    code_str = code_str.replace('.', '')
    try:
        code_num = float(code_str)
    except:
        return 'Other'

    # Map to categories (ordered by frequency/importance)
    if 390 <= code_num < 460 or 785 <= code_num < 786:
        return 'Circulatory'
    elif 460 <= code_num < 520 or 786 <= code_num < 787:
        return 'Respiratory'
    elif 580 <= code_num < 630 or 788 <= code_num < 789:
        return 'Genitourinary'
    elif 240 <= code_num < 280:
        return 'Endocrine'  # Includes Diabetes (250.xx)
    elif 520 <= code_num < 580 or 787 <= code_num < 788:
        return 'Digestive'
    elif 800 <= code_num < 1000:
        return 'Injury'
    elif 780 <= code_num < 800:
        return 'Symptoms_Signs'
    elif 710 <= code_num < 740:
        return 'Musculoskeletal'
    elif 140 <= code_num < 240:
        return 'Neoplasms'
    elif 1 <= code_num < 140:
        return 'Infectious'
    else:
        return 'Other'


def has_diabetes_code(row):
    """
    Check if patient has a diabetes code in any of the 3 diagnoses

    This binary feature indicates whether diabetes was documented in ANY diagnosis
    during the hospitalization. Having a diabetes code means the patient received
    diabetes-specific treatment, which significantly reduces readmission risk.

    Clinical insight from data:
    - No diabetes code: 11.81% readmission rate
    - Has diabetes code: ~7-8% readmission rate

    The POSITION of diabetes (primary/secondary/tertiary) matters less than
    whether it was documented and treated at all.

    Args:
        row: DataFrame row with diag_1, diag_2, diag_3 columns

    Returns:
        int: 0 = No diabetes code, 1 = Has diabetes code in at least one diagnosis
    """
    if is_diabetes(row['diag_1']) or is_diabetes(row['diag_2']) or is_diabetes(row['diag_3']):
        return 1
    else:
        return 0


# =============================================================================
# STEP 3: Apply Grouping to All Diagnosis Columns
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: Apply Grouping to Create New Features")
print("=" * 80)

# Create category columns for each diagnosis
for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    df_cleaned[f'{diag_col}_category'] = df_cleaned[diag_col].apply(categorize_diagnosis)
    print(f"✓ Created: {diag_col}_category")

# Create binary diabetes code indicator
df_cleaned['has_diabetes_code'] = df_cleaned.apply(has_diabetes_code, axis=1)
print(f"✓ Created: has_diabetes_code (0/1)")

# Convert to appropriate data types for memory efficiency
for col in ['diag_1_category', 'diag_2_category', 'diag_3_category']:
    df_cleaned[col] = df_cleaned[col].astype('category')

df_cleaned['has_diabetes_code'] = df_cleaned['has_diabetes_code'].astype('category')

print("\n✓ All features created successfully!")
print("  Total new columns: 4")
print("  - 3 category columns (10 categories + Other)")
print("  - 1 binary diabetes indicator (0/1)")


# =============================================================================
# STEP 4: Analyze Grouped Distribution (After Grouping)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: Grouped Category Distribution Analysis")
print("=" * 80)

def analyze_grouped_distribution(df, diag_cols=['diag_1', 'diag_2', 'diag_3']):
    """
    Analyze the distribution after grouping into categories
    Shows how grouping solves the long-tail problem
    """
    for diag_col in diag_cols:
        cat_col = f'{diag_col}_category'

        print(f"\n{'='*80}")
        print(f"{diag_col.upper()} - Category Distribution")
        print('='*80)

        # Category distribution
        cat_counts = df[cat_col].value_counts()
        total = len(df)

        print(f"\nTotal records: {total:,}")
        print(f"Number of categories: {len(cat_counts)}")
        print(f"\nCategory distribution:")
        print(f"{'Category':<20s} {'Count':>10s} {'Percentage':>12s}")
        print("-" * 45)

        for cat, count in cat_counts.sort_values(ascending=False).items():
            pct = count / total * 100
            print(f"{cat:<20s} {count:>10,} {pct:>11.2f}%")

# Run the analysis
analyze_grouped_distribution(df_cleaned)

# Analyze diabetes code indicator
print(f"\n{'='*80}")
print("HAS DIABETES CODE Distribution")
print('='*80)

diabetes_code_counts = df_cleaned['has_diabetes_code'].value_counts().sort_index()

print(f"\n{'Value':<10s} {'Description':<40s} {'Count':>10s} {'Percentage':>12s}")
print("-" * 75)

descriptions = {
    0: 'No diabetes code in any diagnosis',
    1: 'Has diabetes code (in diag_1/2/3)'
}

for val, count in diabetes_code_counts.items():
    pct = count / len(df_cleaned) * 100
    desc = descriptions.get(val, 'Unknown')
    print(f"{val:<10d} {desc:<40s} {count:>10,} {pct:>11.2f}%")

print(f"\n💡 Clinical Insight:")
print(f"   Having a diabetes code documented means the patient received")
print(f"   diabetes-specific treatment during hospitalization, which")
print(f"   significantly reduces readmission risk (from ~12% to ~8%)")
print(f"\n   Note: Position (primary/secondary/tertiary) has minimal impact")
print(f"   compared to whether diabetes was documented at all.")


# =============================================================================
# STEP 5: Summary Comparison
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: Before vs After Summary")
print("=" * 80)

# Before grouping
original_unique = df_cleaned['diag_1'].nunique()
original_median = df_cleaned['diag_1'].value_counts().median()

# After grouping
grouped_categories = df_cleaned['diag_1_category'].nunique()
grouped_min = df_cleaned['diag_1_category'].value_counts().min()

print("\n📊 BEFORE Grouping (diag_1 example):")
print(f"  • Unique codes: {original_unique:,}")
print(f"  • Median samples per code: {original_median:.0f}")
print(f"  • Problem: Long-tail distribution with many rare codes")

print("\n📊 AFTER Grouping:")
print(f"  • Number of categories: {grouped_categories}")
print(f"  • Minimum samples per category: {grouped_min:,}")
print(f"  • Solution: All categories well-represented!")

print("\n✅ Grouping completed successfully!")
print(f"\nFinal feature set:")
print(f"  • diag_1_category, diag_2_category, diag_3_category")
print(f"    → Categorical with 10 clinical categories + Other")
print(f"  • has_diabetes_code")
print(f"    → Binary (0/1) indicating if diabetes was documented")
print(f"\nTotal: 4 new columns created")
print(f"\nDesign decisions:")
print(f"  ✓ Simplified diabetes to binary (has code or not)")
print(f"    - Position had only 235 samples in diag_1 (too few)")
print(f"    - All positions show similar readmission reduction")
print(f"    - What matters: was diabetes documented and treated?")
print(f"  ✓ No severity columns (96% would be 'Mild', no variance)")
print(f"  ✓ Use existing features for severity: time_in_hospital, num_procedures, etc.")

STEP 1: Original ICD-9 Category Distribution Analysis

DIAG_1 - Original ICD-9 Category Distribution

Total records: 100,114
Number of ICD-9 categories: 20

ICD-9 Category                 Count   Percentage
--------------------------------------------------
Circulatory                   29,782       29.75%
Respiratory                   10,058       10.05%
Digestive                      9,115        9.10%
Other                          8,458        8.45%
Symptoms_Signs                 7,618        7.61%
Injury_Poisoning               6,881        6.87%
Genitourinary                  5,015        5.01%
Musculoskeletal                4,944        4.94%
Neoplasms                      3,299        3.30%
Endocrine_Metabolic            2,903        2.90%
Infectious                     2,585        2.58%
Skin                           2,513        2.51%
Mental                         2,259        2.26%
V_codes                        1,638        1.64%
Nervous                        1,192      

## 12. Age Group Feature Engineering

Analyze current age distribution and create consolidated age groups.

In [17]:
df_cleaned['age'].value_counts()


,count
age,
[70-80),25562
[60-70),22185
[50-60),17102
[80-90),16706
[40-50),9626
[30-40),3765
[90-100),2668
[20-30),1650
[10-20),690


**New Age Groups**
The new age grouping reflects typical diabetes onset patterns and complication risks: Young (0-40) represents early-onset diabetes with unique management challenges, Middle (40-60) captures the peak age for type 2 diabetes diagnosis, Older (60-80) reflects increased comorbidities and complications, and Elderly (80-100) represents the highest risk group with complex multi-system involvement.

In [18]:
# Consolidate age groups from 10 to 4 groups
# Define the mapping: original age group -> new consolidated group
age_group_map = {
    '[0-10)': 'Young',
    '[10-20)': 'Young',
    '[20-30)': 'Young',
    '[30-40)': 'Young',
    '[40-50)': 'Middle',
    '[50-60)': 'Middle',
    '[60-70)': 'Older',
    '[70-80)': 'Older',
    '[80-90)': 'Older',
    '[90-100)': 'Older'
}

# Apply the mapping to create new age_group column
df_cleaned['age_group'] = df_cleaned['age'].map(age_group_map)

# Convert to category type for efficiency
df_cleaned['age_group'] = df_cleaned['age_group'].astype('category')

# Display the distribution
print("Age group distribution:")
print(df_cleaned['age_group'].value_counts().sort_index())

Age group distribution:
age_group
Middle    26728
Older     67121
Young      6265
Name: count, dtype: int64


In [19]:
df_cleaned['age_group'].value_counts()


,count
age_group,
Older,67121
Middle,26728
Young,6265


## 13. Frequency Analysis

Calculate and analyze various frequency distributions.

In [20]:
"""
calcuclating frequencies
"""

# =============================================================================
# Translation dictionaries
# =============================================================================

# Admission Type ID
admission_labels = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: 'NULL',
    7: 'Trauma Center',
    8: 'Not Mapped'
}

# Discharge Disposition ID - most common
discharge_labels = {
    1: 'Discharged to home',
    2: 'Discharged to another hospital',
    3: 'Discharged to SNF',
    6: 'Discharged with home health service',
    11: 'Expired',
    13: 'Hospice/home',
    14: 'Hospice/medical facility',
    18: 'NULL',
    25: 'Not Mapped',
    7: 'Left AMA (Against Medical Advice)'
}

# Admission Source ID - most common
source_labels = {
    1: 'Physician Referral',
    2: 'Clinic Referral',
    4: 'Transfer from hospital',
    5: 'Transfer from SNF',
    7: 'Emergency Room',
    9: 'Not Available',
    17: 'NULL',
    20: 'Not Mapped'
}

# =============================================================================
# General display function
# =============================================================================

def show_frequencies(column_name, labels_dict, title):
    """
    Display frequencies with labels, sorted by size
    """
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    # Count
    counts = df_cleaned[column_name].value_counts()
    total = len(df_cleaned)

    print(f"\nTotal records: {total:,}")
    print(f"Unique codes: {len(counts)}")
    print(f"\nSorted by frequency (highest to lowest):\n")
    print(f"{'#':<3} | {'ID':<3} | {'Description':<50} | {'Count':>10} | {'%':>6}")
    print("-" * 80)

    for rank, (code_id, count) in enumerate(counts.items(), 1):
        pct = count / total * 100
        label = labels_dict.get(code_id, f'Unknown ({code_id})')
        print(f"{rank:<3} | {code_id:<3} | {label:<50} | {count:>10,} | {pct:>5.2f}%")

# =============================================================================
# Execute
# =============================================================================

show_frequencies(
    'admission_type_id',
    admission_labels,
    '1. Admission Type ID'
)

show_frequencies(
    'discharge_disposition_id',
    discharge_labels,
    '2. Discharge Disposition ID'
)

show_frequencies(
    'admission_source_id',
    source_labels,
    '3. Admission Source ID'
)

print("\n" + "=" * 80)
print("✓ Analysis completed!")
print("=" * 80)


1. Admission Type ID

Total records: 100,114
Unique codes: 8

Sorted by frequency (highest to lowest):

#   | ID  | Description                                        |      Count |      %
--------------------------------------------------------------------------------
1   | 1   | Emergency                                          |     52,884 | 52.82%
2   | 3   | Elective                                           |     18,739 | 18.72%
3   | 2   | Urgent                                             |     18,226 | 18.21%
4   | 6   | NULL                                               |      5,227 |  5.22%
5   | 5   | Not Available                                      |      4,690 |  4.68%
6   | 8   | Not Mapped                                         |        320 |  0.32%
7   | 7   | Trauma Center                                      |         18 |  0.02%
8   | 4   | Newborn                                            |         10 |  0.01%

2. Discharge Disposition ID

Total records: 100,

## 14. Newborn Cases Analysis

Identify and analyze newborn cases in the dataset.

In [21]:



# קריאת הדאטה המקורית
df_original = df1

# סינון מקרי newborn
newborns = df_original[df_original['admission_type_id'] == 4].copy()

print(f"\n{'=' * 80}")
print(f"מספר מקרים כולל: {len(newborns)}")
print(f"{'=' * 80}")

print("\n" + "=" * 80)
print("1. מאפיינים דמוגרפיים מפורטים")
print("=" * 80)

print("\n--- גיל (Age) ---")
age_dist = newborns['age'].value_counts().sort_index()
for age_group, count in age_dist.items():
    pct = count / len(newborns) * 100
    print(f"  {age_group}: {count} ({pct:.1f}%)")




מספר מקרים כולל: 10

1. מאפיינים דמוגרפיים מפורטים

--- גיל (Age) ---
  [0-10): 1 (10.0%)
  [40-50): 1 (10.0%)
  [50-60): 1 (10.0%)
  [60-70): 3 (30.0%)
  [70-80): 2 (20.0%)
  [80-90): 2 (20.0%)


ניתן לראות שיש שגיאה בקידוד של הסיבה לאישפוז כי קודדו כיילודים למרות שהם בגילאים מבוגרים יותר. לכן סיבת הקידוד תהפוך ללא ידועה

In [22]:
print("=" * 80)
print("תיקון שגיאת קידוד: Newborn → Not Available")
print("=" * 80)

# זיהוי המקרים
newborn_errors = df_cleaned['admission_type_id'] == 4

print(f"\nמקרים שמסומנים כ-Newborn: {newborn_errors.sum()}")
print(f"גילאי המקרים:")
print(df_cleaned[newborn_errors]['age'].value_counts().sort_index())

# תיקון
df_cleaned.loc[newborn_errors, 'admission_type_id'] = 5

print(f"\n✓ {newborn_errors.sum()} מקרים תוקנו מ-4 (Newborn) ל-5 (Not Available)")
print("✓ הרשומות נשמרו - רק הקוד תוקן")

תיקון שגיאת קידוד: Newborn → Not Available

מקרים שמסומנים כ-Newborn: 10
גילאי המקרים:
age
[0-10)     1
[40-50)    1
[50-60)    1
[60-70)    3
[70-80)    2
[80-90)    2
Name: count, dtype: Int64

✓ 10 מקרים תוקנו מ-4 (Newborn) ל-5 (Not Available)
✓ הרשומות נשמרו - רק הקוד תוקן


## 15. Admission Type Mapping

Map admission type IDs to meaningful categories.

In [23]:


# החלפנו את תהליך קריאת הקובץ במילון מוכן מראש
admission_type_dict = {
    1: 'Emergency',
    2: 'Urgent',
    3: 'Elective',
    4: 'Newborn',
    5: 'Not Available',
    6: 'NULL',
    7: 'Trauma Center',
    8: 'Not Mapped'
}

#הסבת הקטגוריה של טראומה ל-emergency
df_cleaned['admission_grouped'] = df_cleaned['admission_type_id'].apply(
    lambda x: 1 if x == 7 else x  # Convert Trauma (7) to Emergency (1)
)

# שמירה על הקטגוריות של חירום, דחוף, אלקטיבי -
main_categories = [1, 2, 3]
df_cleaned['admission_grouped'] = df_cleaned['admission_type_id'].apply(
    lambda x: x if x in main_categories else 5
)

df_cleaned['admission_grouped'] = df_cleaned['admission_grouped'].astype('category')




## 16. Medical Specialty Grouping

Analyze and group medical specialties into major categories.

In [24]:
# בדיקה - כמה התמחויות יש מתחת ל-250 מקרים
print("=" * 70)
print("ניתוח התמחויות לפי כמות מקרים")
print("=" * 70)

specialty_counts = df_cleaned['medical_specialty'].value_counts()


below_250 = specialty_counts[specialty_counts < 250].sort_values()
for specialty, count in below_250.items():
    print(f"{specialty:40s} : {count:3}")

print(f"\nסה\"כ: {len(below_250)} התמחויות עם פחות מ-250 מקרים")

ניתוח התמחויות לפי כמות מקרים
Speech                                   :   1
SportsMedicine                           :   1
Perinatology                             :   1
Surgery-PlasticwithinHeadandNeck         :   1
Proctology                               :   1
Dermatology                              :   1
Pediatrics-InfectiousDiseases            :   1
Neurophysiology                          :   1
Psychiatry-Addictive                     :   1
Resident                                 :   2
Pediatrics-AllergyandImmunology          :   3
Pediatrics-EmergencyMedicine             :   3
Pediatrics-Hematology-Oncology           :   4
Dentistry                                :   4
DCPTEAM                                  :   5
Psychiatry-Child/Adolescent              :   7
Cardiology-Pediatric                     :   7
AllergyandImmunology                     :   7
Endocrinology-Metabolism                 :   8
Surgery-Pediatric                        :   8
Pediatrics-Neurology          

In [25]:
#קיבוץ ההתמחויות לפי קבוצות

SPECIALTY_MAPPING= {
    'InternalMedicine': ['InternalMedicine', 'Endocrinology', 'Endocrinology-Metabolism', 'Pediatrics-Endocrinology', 'Hospitalist', 'InfectiousDiseases'],
    'Emergency': ['Emergency/Trauma', 'Pediatrics-EmergencyMedicine', 'Pediatrics-CriticalCare'],
    'FamilyPractice': ['Family/GeneralPractice'],
    'Cardiology': ['Cardiology', 'Cardiology-Pediatric', 'Surgery-Cardiovascular', 'Surgery-Cardiovascular/Thoracic'],
    'Surgery': ['Surgery-General', 'Surgery-Vascular', 'Surgery-Neuro', 'Surgery-Thoracic', 'Surgery-Plastic', 'Surgery-Colon&Rectal', 'Surgery-Maxillofacial', 'Surgery-Pediatric', 'Surgery-PlasticwithinHeadandNeck', 'Surgeon', 'SurgicalSpecialty'],
    'Orthopedics': ['Orthopedics', 'Orthopedics-Reconstructive'],
    'Psychiatry': ['Psychiatry', 'Psychiatry-Child/Adolescent', 'Psychiatry-Addictive', 'Psychology'],
    'ObGyn': ['ObstetricsandGynecology', 'Gynecology', 'Obsterics&Gynecology-GynecologicOnco', 'Obstetrics'],
    'Oncology': ['Oncology', 'Hematology/Oncology', 'Hematology', 'Pediatrics-Hematology-Oncology'],
    'Pulmonology': ['Pulmonology', 'Pediatrics-Pulmonology'],
    'Nephrology': ['Nephrology'],
    'Radiology': ['Radiologist', 'Radiology'],
    'Urology': ['Urology'],
    'Gastroenterology': ['Gastroenterology'],
    'Neurology': ['Neurology', 'Neurophysiology', 'Pediatrics-Neurology'],
    'Rehabilitation': ['PhysicalMedicineandRehabilitation'],
    'GeneralPediatrics': ['Pediatrics']
}


def map_specialty(specialty):
    """ממפה התמחות לקטגוריה"""
    if pd.isna(specialty) or specialty == '?':
        return 'Missing'

    for category, specialties in SPECIALTY_MAPPING.items():
        if specialty in specialties:
            return category

    return 'Other'

df_cleaned['specialty_grouped'] = df_cleaned['medical_specialty'].apply(map_specialty)
print(df_cleaned['specialty_grouped'].value_counts())

specialty_grouped
Missing              49129
InternalMedicine     14708
Emergency             7539
FamilyPractice        7302
Cardiology            6043
Surgery               4314
Orthopedics           2625
Nephrology            1544
Radiology             1182
Psychiatry             962
Pulmonology            881
ObGyn                  771
Urology                684
Oncology               609
Gastroenterology       550
Other                  414
Rehabilitation         391
GeneralPediatrics      253
Neurology              213
Name: count, dtype: int64


In [26]:

# List of specialties to KEEP (those with counts > 1000)
# This includes 'Missing' as a critical predictor.
SPECIALTIES_TO_KEEP = [
    'Missing', 'InternalMedicine', 'Emergency', 'FamilyPractice',
    'Cardiology', 'Surgery', 'Orthopedics', 'Nephrology', 'Radiology'
]
OTHER_LABEL = 'Other_Rare_Specialty'

# Assuming df_cleaned is your DataFrame and specialty_grouped is the column
col = 'specialty_grouped'
new_col = 'specialty_grouped'

# 1. Apply mapping logic: If specialty is in the KEEP list, keep it. Otherwise, label it as 'Other_Rare_Specialty'.
df_cleaned[new_col] = np.where(
    df_cleaned[col].isin(SPECIALTIES_TO_KEEP),
    df_cleaned[col], OTHER_LABEL)


print("✓ Cardinality for specialty was reduced to 10 levels.")


✓ Cardinality for specialty was reduced to 10 levels.


discharge_disposition_id	description



1	Discharged to home
2	Discharged/transferred to another short term hospital
3	Discharged/transferred to SNF
4	Discharged/transferred to ICF
5	Discharged/transferred to another type of inpatient care institution
6	Discharged/transferred to home with home health service
7	Left AMA
8	Discharged/transferred to home under care of Home IV provider
9	Admitted as an inpatient to this hospital
10	Neonate discharged to another hospital for neonatal aftercare
11	Expired
12	Still patient or expected to return for outpatient services
13	Hospice / home
14	Hospice / medical facility
15	Discharged/transferred within this institution to Medicare approved swing bed
16	Discharged/transferred/referred another institution for outpatient services
17	Discharged/transferred/referred to this institution for outpatient services
18	NULL
19	Expired at home. Medicaid only, hospice.
20	Expired in a medical facility. Medicaid only, hospice.
21	Expired, place unknown. Medicaid only, hospice.
22	Discharged/transferred to another rehab fac including rehab units of a hospital .
23	Discharged/transferred to a long term care hospital.
24	Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.
25	Not Mapped
26	Unknown/Invalid
30	Discharged/transferred to another Type of Health Care Institution not Defined Elsewhere
27	Discharged/transferred to a federal health care facility.
28	Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital
29	Discharged/transferred to a Critical Access Hospital (CAH).


## 17. Discharge Disposition Grouping

Categorize discharge destinations for patients.

In [27]:
# התפלגות discharge_disposition_id
print("=" * 70)
print("התפלגות Discharge Disposition")
print("=" * 70)

discharge_dist = df_cleaned['discharge_disposition_id'].value_counts().sort_index()

for disp_id, count in discharge_dist.items():
    pct = (count / len(df_cleaned)) * 100
    print(f"ID {disp_id:2d}: {count:6,} ({pct:5.2f}%)")

print(f"\nסה\"כ קטגוריות שונות: {df_cleaned['discharge_disposition_id'].nunique()}")

התפלגות Discharge Disposition
ID  1: 60,234 (60.17%)
ID  2:  2,128 ( 2.13%)
ID  3: 13,954 (13.94%)
ID  4:    815 ( 0.81%)
ID  5:  1,184 ( 1.18%)
ID  6: 12,902 (12.89%)
ID  7:    623 ( 0.62%)
ID  8:    108 ( 0.11%)
ID  9:     21 ( 0.02%)
ID 10:      6 ( 0.01%)
ID 12:      3 ( 0.00%)
ID 13:    399 ( 0.40%)
ID 14:    372 ( 0.37%)
ID 15:     63 ( 0.06%)
ID 16:     11 ( 0.01%)
ID 17:     14 ( 0.01%)
ID 18:  3,691 ( 3.69%)
ID 22:  1,993 ( 1.99%)
ID 23:    412 ( 0.41%)
ID 24:     48 ( 0.05%)
ID 25:    989 ( 0.99%)
ID 27:      5 ( 0.00%)
ID 28:    139 ( 0.14%)

סה"כ קטגוריות שונות: 23


In [28]:
# Grouping discharge disposition categories
def group_discharge(disp_id):
    # Home - including outpatient follow-up
    if disp_id in [1, 6, 8, 16, 17]:
        return 'Home'

    # Transfer to Facility - hospitalization/institutional care
    elif disp_id in [2, 3, 4, 5, 10, 15, 22, 23, 24, 27, 28, 29, 30]:
        return 'Transfer_to_Facility'

    # Hospice
    elif disp_id in [13, 14]:
        return 'Hospice'

    # Left AMA
    elif disp_id == 7:
        return 'Left_AMA'

    # Unknown
    elif disp_id in [18, 25, 26, 9, 12]:
        return 'Unknown'

    else:
        return 'Other'

# Update the column
df_cleaned['discharge_grouped'] = df_cleaned['discharge_disposition_id'].apply(group_discharge)

# Check new distribution
print("=" * 70)
print("Discharge Grouped Distribution - Updated")
print("=" * 70)

discharge_counts = df_cleaned['discharge_grouped'].value_counts()
for category, count in discharge_counts.items():
    pct = (count / len(df_cleaned)) * 100
    print(f"{category:25s}: {count:7,} ({pct:5.2f}%)")

print(f"\nTotal categories: {df_cleaned['discharge_grouped'].nunique()}")

# Verify no Other remains
if 'Other' in discharge_counts.index:
    print(f"\n⚠️ Still have {discharge_counts['Other']} cases in Other")
else:
    print("\n✓ All cases successfully grouped")

Discharge Grouped Distribution - Updated
Home                     :  73,269 (73.19%)
Transfer_to_Facility     :  20,747 (20.72%)
Unknown                  :   4,704 ( 4.70%)
Hospice                  :     771 ( 0.77%)
Left_AMA                 :     623 ( 0.62%)

Total categories: 5

✓ All cases successfully grouped


 Admission Source Distribution

## Admission Source ID Codes

| ID | Description |
|---|---|
| 1 | Physician Referral |
| 2 | Clinic Referral |
| 3 | HMO Referral |
| 4 | Transfer from a hospital |
| 5 | Transfer from a Skilled Nursing Facility (SNF) |
| 6 | Transfer from another health care facility |
| 7 | Emergency Room |
| 8 | Court/Law Enforcement |
| 9 | Not Available |
| 10 | Transfer from critical access hospital |
| 11 | Normal Delivery |
| 12 | Premature Delivery |
| 13 | Sick Baby |
| 14 | Extramural Birth |
| 15 | Not Available |
| 17 | NULL |
| 18 | Transfer From Another Home Health Agency |
| 19 | Readmission to Same Home Health Agency |
| 20 | Not Mapped |
| 21 | Unknown/Invalid |
| 22 | Transfer from hospital inpt/same fac reslt in a sep claim |
| 23 | Born inside this hospital |
| 24 | Born outside this hospital |
| 25 | Transfer from Ambulatory Surgery Center |
| 26 | Transfer from Hospice |

## 18. Admission Source Grouping

Group admission sources into meaningful categories.

In [29]:
# Admission_source_id distribution
print("=" * 70)
print("Admission Source Distribution")
print("=" * 70)

admission_dist = df_cleaned['admission_source_id'].value_counts().sort_index()

for source_id, count in admission_dist.items():
    pct = (count / len(df_cleaned)) * 100
    print(f"ID {source_id:2d}: {count:6,} ({pct:5.2f}%)")

print(f"\nTotal distinct categories: {df_cleaned['admission_source_id'].nunique()}")

Admission Source Distribution
ID  1: 29,322 (29.29%)
ID  2:  1,083 ( 1.08%)
ID  3:    185 ( 0.18%)
ID  4:  3,132 ( 3.13%)
ID  5:    814 ( 0.81%)
ID  6:  2,244 ( 2.24%)
ID  7: 56,363 (56.30%)
ID  8:     15 ( 0.01%)
ID  9:    125 ( 0.12%)
ID 10:      8 ( 0.01%)
ID 11:      2 ( 0.00%)
ID 13:      1 ( 0.00%)
ID 14:      2 ( 0.00%)
ID 17:  6,645 ( 6.64%)
ID 20:    159 ( 0.16%)
ID 22:     12 ( 0.01%)
ID 25:      2 ( 0.00%)

Total distinct categories: 17


In [30]:
birth_count = df_cleaned['admission_source_id'].isin([11, 12, 13, 14, 23, 24]).sum()
print(f"Birth-related: {birth_count} cases  ")

Birth-related: 5 cases  


In [31]:
# Check for babies by age
print("=" * 70)
print("Checking for babies in the data")
print("=" * 70)

# How many are in age group [0-10)
babies = df_cleaned[df_cleaned['age_group'] == 5]  # age_group=5 represents [0-10)
print(f"Total in age group [0-10): {len(babies)} ({len(babies)/len(df_cleaned)*100:.2f}%)")

# How many of them have birth-related admission_source
birth_source_ids = [11, 12, 13, 14, 23, 24]
babies_from_birth = df_cleaned[
    (df_cleaned['age_group'] == 5) &
    (df_cleaned['admission_source_id'].isin(birth_source_ids))
]
print(f"Among them, those with birth-related admission source: {len(babies_from_birth)}")

# What is the admission_source distribution for the remaining babies?
print("\nAdmission_source distribution for babies [0-10):")
if len(babies) > 0:
    babies_admission = babies['admission_source_id'].value_counts()
    for source_id, count in babies_admission.items():
        print(f"  ID {source_id}: {count} babies")
else:
    print("  No babies in the data")

# Are all birth-related cases babies?
print("\nAre all birth-related cases babies?")
birth_cases = df_cleaned[df_cleaned['admission_source_id'].isin(birth_source_ids)]
if len(birth_cases) > 0:
    print(f"Total birth-related cases: {len(birth_cases)}")
    print(f"Age distribution:")
    print(birth_cases['age_group'].value_counts().sort_index())

Checking for babies in the data
Total in age group [0-10): 0 (0.00%)
Among them, those with birth-related admission source: 0

Admission_source distribution for babies [0-10):
  No babies in the data

Are all birth-related cases babies?
Total birth-related cases: 5
Age distribution:
age_group
Middle    0
Older     4
Young     1
Name: count, dtype: int64


In [32]:
"""
Grouping Admission Source ID with map
"""

print("=" * 80)
print("Admission Source Grouping")
print("=" * 80)

# =============================================================================
# Direct mapping of codes to categories
# =============================================================================

admission_source_mapping = {
    # Emergency Room
    7: 'Emergency',

    # Referral (physician, clinic, HMO, court)
    1: 'Referral',
    2: 'Referral',
    3: 'Referral',
    8: 'Referral',  # Court/Law Enforcement

    # Transfer from Facility
    4: 'Transfer_from_Facility',   # from hospital
    5: 'Transfer_from_Facility',   # from SNF
    6: 'Transfer_from_Facility',   # from other health facility
    10: 'Transfer_from_Facility',  # from critical access hospital
    18: 'Transfer_from_Facility',  # from home health agency
    22: 'Transfer_from_Facility',  # internal transfer
    25: 'Transfer_from_Facility',  # from surgery center
    26: 'Transfer_from_Facility',  # from Hospice

    # Readmission from Home Health Agency
    19: 'Readmission_HHA',

    # Unknown (including Birth-related - coding errors!)
    9: 'Unknown',
    17: 'Unknown',
    20: 'Unknown',
    21: 'Unknown',
    11: 'Unknown',  # Birth - coding error
    12: 'Unknown',  # Birth - coding error
    13: 'Unknown',  # Birth - coding error
    14: 'Unknown',  # Birth - coding error
    23: 'Unknown',  # Birth - coding error
    24: 'Unknown',  # Birth - coding error
}

# mapping the codes
df_cleaned['admission_source_grouped'] = df_cleaned['admission_source_id'].map(
    admission_source_mapping
).fillna('Other')

# =============================================================================
# Distribution
# =============================================================================
print("\nAdmission Source Grouped Distribution:")
print("=" * 80)

grouped_counts = df_cleaned['admission_source_grouped'].value_counts()
total = len(df_cleaned)

for category, count in grouped_counts.items():
    pct = count / total * 100
    print(f"  {category:25s}: {count:7,} ({pct:5.2f}%)")

print(f"\nTotal categories: {df_cleaned['admission_source_grouped'].nunique()}")



Admission Source Grouping

Admission Source Grouped Distribution:
  Emergency                :  56,363 (56.30%)
  Referral                 :  30,605 (30.57%)
  Unknown                  :   6,934 ( 6.93%)
  Transfer_from_Facility   :   6,212 ( 6.20%)

Total categories: 4


## 19. Dataset Review and Validation

Review the cleaned dataset structure and contents.

In [33]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 61 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   encounter_id              100114 non-null  int64   
 1   patient_nbr               100114 non-null  int64   
 2   race                      100114 non-null  string  
 3   gender                    100114 non-null  string  
 4   age                       100114 non-null  string  
 5   admission_type_id         100114 non-null  int64   
 6   discharge_disposition_id  100114 non-null  int64   
 7   admission_source_id       100114 non-null  int64   
 8   time_in_hospital          100114 non-null  int64   
 9   payer_code                100114 non-null  string  
 10  medical_specialty         100114 non-null  string  
 11  num_lab_procedures        100114 non-null  int64   
 12  num_procedures            100114 non-null  int64   
 13  num_medications           100114 n

In [34]:
df_cleaned

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,payer_grouped,diag_1_category,diag_2_category,diag_3_category,has_diabetes_code,age_group,admission_grouped,specialty_grouped,discharge_grouped,admission_source_grouped
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,?,...,Unknown,Other,Other,Other,1,Young,5,InternalMedicine,Unknown,Referral
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,?,...,Unknown,Endocrine,Other,Endocrine,1,Young,1,Missing,Home,Emergency
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,?,...,Unknown,Other,Endocrine,Other,1,Young,1,Missing,Home,Emergency
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,?,...,Unknown,Infectious,Other,Circulatory,1,Young,1,Missing,Home,Emergency
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,?,...,Unknown,Neoplasms,Neoplasms,Endocrine,1,Middle,1,Missing,Home,Emergency
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),1,3,7,3,MC,...,Medicare,Other,Other,Circulatory,1,Older,1,Missing,Transfer_to_Facility,Emergency
101762,443847782,74694222,AfricanAmerican,Female,[80-90),1,4,5,5,MC,...,Medicare,Digestive,Endocrine,Digestive,0,Older,1,Missing,Transfer_to_Facility,Transfer_from_Facility
101763,443854148,41088789,Caucasian,Male,[70-80),1,1,7,1,MC,...,Medicare,Infectious,Genitourinary,Other,0,Older,1,Missing,Home,Emergency
101764,443857166,31693671,Caucasian,Female,[80-90),2,3,7,10,MC,...,Medicare,Injury,Other,Injury,0,Older,2,Surgery,Transfer_to_Facility,Emergency


## 20. Medication Analysis and Selection

Analyze medication usage patterns and select relevant medication features.

In [35]:
"""
identifing and removing empty medication columns (0% usage), removing columns where no patients
received the medication (all values are "No").
"""

# List of medication columns to check
# These represent 23 different diabetes medications tracked in the dataset
medication_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

print("Checking for Empty Medication Columns (0% Usage)")
print("=" * 70)

# Store original shape
original_shape = df_cleaned.shape

# Find columns with 0% usage
empty_columns = []

for col in medication_columns:
    if col in df_cleaned.columns:
        # Check if all values are "No" (meaning no patients received this medication)
        usage_count = (df_cleaned[col] != 'No').sum()

        if usage_count == 0:
            empty_columns.append(col)
            print(f"❌ {col:<30} - 0% usage (all values: No)")

print("=" * 70)
print(f"\nTotal empty columns found: {len(empty_columns)}")

if len(empty_columns) > 0:
    print("\nColumns to be removed:")
    for col in empty_columns:
        print(f"  - {col}")

    # Drop the empty columns from df_cleaned
    print("\n" + "=" * 70)
    print("Removing empty columns from df_cleaned...")
    print("=" * 70)

    df_cleaned.drop(columns=empty_columns, inplace=True)

    print(f"\n✓ Successfully removed {len(empty_columns)} empty medication columns")

    # Save the cleaned dataset
    output_filename = 'diabetic_data_cleaned.csv'
    df_cleaned.to_csv(output_filename, index=False)
    print(f"✓ Cleaned dataset saved as: {output_filename}")
else:
    print("\n✓ No empty columns found!")

# Print dataset summary
print("\n" + "=" * 70)
print("Dataset Summary:")
print("=" * 70)
print(f"Original shape:  {original_shape[0]} rows × {original_shape[1]} columns")
print(f"Cleaned shape:   {df_cleaned.shape[0]} rows × {df_cleaned.shape[1]} columns")
print(f"Columns removed: {len(empty_columns)}")

Checking for Empty Medication Columns (0% Usage)
❌ examide                        - 0% usage (all values: No)
❌ citoglipton                    - 0% usage (all values: No)

Total empty columns found: 2

Columns to be removed:
  - examide
  - citoglipton

Removing empty columns from df_cleaned...

✓ Successfully removed 2 empty medication columns
✓ Cleaned dataset saved as: diabetic_data_cleaned.csv

Dataset Summary:
Original shape:  100114 rows × 61 columns
Cleaned shape:   100114 rows × 59 columns
Columns removed: 2


In [36]:
for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'string':
        question_marks_count = (df_cleaned[col] == '?').sum()
        #features with ?
        if question_marks_count > 0:
            print(f"feature '{col}': {question_marks_count} ?")

feature 'payer_code': 39591 ?
feature 'medical_specialty': 49129 ?
feature 'diag_1': 21 ?
feature 'diag_2': 358 ?
feature 'diag_3': 1421 ?


## 21. Final Dataset Preparation (df_new)

Create the final dataset with selected features and optimized data types.

In [37]:
# ===========================
# Creating df_new - Only Columns for ML Model
# ===========================

"""
Model DataFrame Creation
------------------------
This script creates a clean DataFrame (df_new) containing only the columns
needed for machine learning modeling. It excludes raw/ungrouped columns and
converts categorical features to the 'category' dtype for efficiency.
"""

print("=" * 80)
print("📊 CREATING MODEL DATAFRAME")
print("=" * 80)

# =============================================================================
# Define Columns for the Model
# =============================================================================

model_columns = [
    # Identifiers (will be dropped before training)
    'patient_nbr',
    'encounter_id',

    # Demographics
    'race',
    'gender',
    'age_group',

    # Grouped codes (not the original ones!)
    'admission_grouped',
    'specialty_grouped',
    'admission_source_grouped',
    'payer_grouped',
    'discharge_grouped',

    # Medical metrics
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_diagnoses',

    # Medical history
    'number_outpatient',
    'number_emergency',
    'number_inpatient',

    # Diagnoses (category + severity)
    'diag_1_category',
    'diag_2_category',
    'diag_3_category',
    'has_diabetes_code',

    # A1C test results
    'A1Cresult',

    # Diabetes medications (7 columns)
    'metformin',
    'glimepiride',
    'glipizide',
    'glyburide',
    'pioglitazone',
    'rosiglitazone',
    'insulin',

    # Treatment changes
    'change',
    'diabetesMed',

    # Target variable
    'readmitted_binary'
]

# =============================================================================
# Create df_new
# =============================================================================

# Check that all columns exist
missing_cols = [col for col in model_columns if col not in df_cleaned.columns]
if missing_cols:
    print(f"\n⚠️  Missing columns: {missing_cols}")
    # Remove missing columns from the list
    model_columns = [col for col in model_columns if col in df_cleaned.columns]

# Create new DataFrame
df_new = df_cleaned[model_columns].copy()

print(f"✅ Created df_new with {len(model_columns)} columns")

# =============================================================================
# Convert All Categorical Columns to 'category' Dtype
# =============================================================================
print("\n" + "=" * 80)
print("🔄 CONVERTING COLUMNS TO 'category' DTYPE")
print("=" * 80)

# Identify columns that should be 'category'
string_object_cols = df_new.select_dtypes(include=['string', 'object']).columns.tolist()

print(f"\nConverting {len(string_object_cols)} columns from string/object to category:")
for col in string_object_cols:
    df_new[col] = df_new[col].astype('category')
    print(f"  ✓ {col}")

# =============================================================================
# Summary
# =============================================================================
print("\n" + "=" * 80)
print("✅ df_new CREATED - READY FOR MODELING")
print("=" * 80)

print(f"\nDataFrame Summary:")
print(f"  • Rows:    {len(df_new):,}")
print(f"  • Columns: {len(df_new.columns)}")
print(f"  • Memory:  {df_new.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 80)
print("📋 COLUMN TYPES IN df_new:")
print("=" * 80)

# Corrected dtype grouping
dtype_summary = {}
for col in df_new.columns:
    dtype_name = str(df_new[col].dtype)
    dtype_summary[dtype_name] = dtype_summary.get(dtype_name, 0) + 1

for dtype, count in sorted(dtype_summary.items(), key=lambda x: x[1], reverse=True):
    print(f"  {dtype:15s}: {count:3d} columns")

print("\n" + "=" * 80)
print("📝 COLUMNS EXCLUDED (not in model):")
print("=" * 80)

excluded = [col for col in df_cleaned.columns if col not in model_columns]
print(f"\nTotal excluded: {len(excluded)} columns")
print("\nExcluded columns:")
for col in excluded:
    print(f"  • {col}")

print("\n" + "=" * 80)
print("💡 NOTES ON EXCLUDED COLUMNS:")
print("=" * 80)
print("  • patient_nbr - Identifier, not a feature")
print("  • encounter_id - Identifier, not a feature")
print("  • admission_type_id, discharge_disposition_id, admission_source_id")
print("    → Grouped versions are included instead")
print("  • payer_code, medical_specialty")
print("    → Grouped versions are included instead")
print("  • diag_1, diag_2, diag_3")
print("    → Category + severity versions are included instead")
print("  • age")
print("    → age_group is included instead")
print("  • readmitted, change_binary")
print("    → Processed versions are included instead")
print("=" * 80)

📊 CREATING MODEL DATAFRAME
✅ Created df_new with 33 columns

🔄 CONVERTING COLUMNS TO 'category' DTYPE

Converting 15 columns from string/object to category:
  ✓ race
  ✓ gender
  ✓ specialty_grouped
  ✓ admission_source_grouped
  ✓ payer_grouped
  ✓ discharge_grouped
  ✓ metformin
  ✓ glimepiride
  ✓ glipizide
  ✓ glyburide
  ✓ pioglitazone
  ✓ rosiglitazone
  ✓ insulin
  ✓ change
  ✓ diabetesMed

✅ df_new CREATED - READY FOR MODELING

DataFrame Summary:
  • Rows:    100,114
  • Columns: 33
  • Memory:  10.61 MB

📋 COLUMN TYPES IN df_new:
  category       :  22 columns
  int64          :  10 columns
  bool           :   1 columns

📝 COLUMNS EXCLUDED (not in model):

Total excluded: 26 columns

Excluded columns:
  • age
  • admission_type_id
  • discharge_disposition_id
  • admission_source_id
  • payer_code
  • medical_specialty
  • diag_1
  • diag_2
  • diag_3
  • repaglinide
  • nateglinide
  • chlorpropamide
  • acetohexamide
  • tolbutamide
  • acarbose
  • miglitol
  • troglitazon

In [38]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   patient_nbr               100114 non-null  int64   
 1   encounter_id              100114 non-null  int64   
 2   race                      100114 non-null  category
 3   gender                    100114 non-null  category
 4   age_group                 100114 non-null  category
 5   admission_grouped         100114 non-null  category
 6   specialty_grouped         100114 non-null  category
 7   admission_source_grouped  100114 non-null  category
 8   payer_grouped             100114 non-null  category
 9   discharge_grouped         100114 non-null  category
 10  time_in_hospital          100114 non-null  int64   
 11  num_lab_procedures        100114 non-null  int64   
 12  num_procedures            100114 non-null  int64   
 13  num_medications           100114 n

## 22. Save Prepared Dataset

Export the cleaned and prepared dataset to Google Drive.

In [39]:
# ===========================
# Saving df_new to Pickle File in Google Drive
# ===========================


print("=" * 80)
print("💾 SAVING df_new TO PICKLE FILE")
print("=" * 80)

# Google Drive path configuration
base_drive_path = '/content/drive/My Drive/'
filename = 'df_after_prep.pkl'
full_path = os.path.join(base_drive_path, filename)

# Save the DataFrame
print(f"\nSaving to: {full_path}")
print("Please wait...")

df_new.to_pickle(full_path)

# Verification
print(f"\n✅ SAVED SUCCESSFULLY!")
print("=" * 80)
print(f"File Details:")
print(f"  • Filename:  {filename}")
print(f"  • Location:  {base_drive_path}")
print(f"  • Rows:      {len(df_new):,}")
print(f"  • Columns:   {len(df_new.columns)}")

# Check file size
file_size = os.path.getsize(full_path) / 1024**2
print(f"  • File size: {file_size:.2f} MB")



💾 SAVING df_new TO PICKLE FILE

Saving to: /content/drive/My Drive/df_after_prep.pkl
Please wait...

✅ SAVED SUCCESSFULLY!
File Details:
  • Filename:  df_after_prep.pkl
  • Location:  /content/drive/My Drive/
  • Rows:      100,114
  • Columns:   33
  • File size: 10.60 MB


In [40]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100114 entries, 0 to 101765
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   patient_nbr               100114 non-null  int64   
 1   encounter_id              100114 non-null  int64   
 2   race                      100114 non-null  category
 3   gender                    100114 non-null  category
 4   age_group                 100114 non-null  category
 5   admission_grouped         100114 non-null  category
 6   specialty_grouped         100114 non-null  category
 7   admission_source_grouped  100114 non-null  category
 8   payer_grouped             100114 non-null  category
 9   discharge_grouped         100114 non-null  category
 10  time_in_hospital          100114 non-null  int64   
 11  num_lab_procedures        100114 non-null  int64   
 12  num_procedures            100114 non-null  int64   
 13  num_medications           100114 n